In [6]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math 
%matplotlib qt5

QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-amr'


#### Defining functions part 1

In [7]:
def get_column_average(bags_path):
  point_array = ["1","2","3","4","5","6","7","7.1","7.2","7.3","7.4","7.5","8","8.1","8.2","8.3","10.1","10.2"]
  X = []
  Y = []
  Z = []
  xyz_list = {}
  for point in point_array:
    csv_file_path=bags_path+f"point_{point}/hedgehog_pos/hedgehog_pos.csv"
    # # Read the CSV file (replace 'data.csv' with your file path)
    df = pd.read_csv(csv_file_path,usecols=['x_m','y_m','z_m'])
    # # Calculate column averages
    column_averages = df.mean()
    xyz_list[f"{point}"] = {
       "x":np.round(column_averages['x_m'],4),
       "y":np.round(column_averages['y_m'],4),
       "z":np.round(column_averages['z_m'],4)
    }
    # X.append(np.round(column_averages['x_m'],4))
    # Y.append(np.round(column_averages['y_m'],4))
    # Z.append(np.round(column_averages['z_m'],4))
  #return [X,Y,Z]
  return xyz_list
def plot_graph(X,Y):
    fig, ax = plt.subplots()
    plt.grid()
    ax.plot(X,Y, 'ro')
    for x,y,p in zip(X, Y,point_array):                                       
        ax.annotate(f"{p}:({x}, {y})", xy=(x,y), xytext=(x, y)) 
    plt.show()


#### Create real points for comparison

The real points positions were generated by the graph.cpp file.

In [8]:
## the cpp code which generates the graph considers the positions 1.2,2.2,3.2, and 4.2 the mobile beacon positions, and 1,2,3, and 4 the fixed beacon position.
# however in the bags, the points 1,2,3 and 4 are the positions of the mobile beacons.
# So to make the graph easier to understand, we are going to remove points 1,2,3 and 4 from the real graph, and 
# change the names of points 1.2,2.2,3.2, and 4.2 to 1,2,3, and 4, respectively.


X_real = [0, 0, 10.25, 10.25, 10.25, 10.25, 0, 0, 0, 5.096, 4.87, 4.87, 4.87, 4.87, 4.87, 4.87, 10.25, 12.25, 14.25, 27.788, -1.8, -3.6]
Y_real = [0, 0.042, 0.019, 0.046, 3.748, 3.7435, 3.749, 3.7445, 1.91, 3.152, 1.119, 7.3935, -0.406, 1.909, 3.794, 5.2935, 1.909, 1.909, 1.909, 1.909, 1.909, 1.909]
P_real = [1, 1.2, 2, 2.2, 3, 3.2, 4, 4.2, 5, 6, 7, 7.1, 7.2, 7.3, 7.4, 7.5, 8, 8.1, 8.2, 8.3, 10.1, 10.2]

real_points = {}
i=0
points_to_exclude = [1,2,3,4]
point_to_change_name = [1.2,2.2,3.2,4.2]
for point in P_real:
    if point not in points_to_exclude:
        if point in point_to_change_name:
            real_points[f'{int(point)}']= {"x":X_real[i],"y":Y_real[i]} 
        else:
            real_points[f'{point}']= {"x":X_real[i],"y":Y_real[i]} 
    i = i + 1

# points_to_exclude = [1,2,3,4]
# for i in points_to_exclude:
#     temp_index = P_real.index(i)
#     del P_real[temp_index]
#     del X_real[temp_index]
#     del Y_real[temp_index]


# point_to_change_name = [1.2,2.2,3.2,4.2]
# for i in point_to_change_name:
#     temp_index = P_real.index(i)
#     P_real[temp_index] = int(P_real[temp_index])

#### Reading data from bags

We define a list of points which match the real points list.
However, they only match after the removal of points 1.2,2.2,3.2, and 4.2 from the real points list.
Then, we read data from the bags:
- Bag1: we recorded the marvelmind program result after passing every fixed beacon position.
- Bag2: we recorded the marvelmind program result afte the fixed beacons distance were calculated by the marvelmind program.


In [9]:
point_array = ["1","2","3","4","5","6","7","7.1","7.2","7.3","7.4","7.5","8","8.1","8.2","8.3","10.1","10.2"]
# points 1,2,3 and 4, are not the fixed beacons point. They are the mobile beacons placed close.
# The comparison will be done between 1 marvelmind and 1.2 real (mobile beacon pose on map).
bags_path="/persistent/scripts/bags/correct_better_zone_marvelmind_bag_data_with_distances/"
#[X_with_distance,Y_with_distance,Z_with_distance] = get_column_average(bags_path)
marvel_with_distance_positions = get_column_average(bags_path)
bags_path="/persistent/scripts/bags/correct_better_zone_marvelmind_bag_data_without_distances/"
#[X_without_distance,Y_without_distance,Z_without_distance] = get_column_average(bags_path)
marvel_without_distance_positions = get_column_average(bags_path)

#### Defining functions part 2

Define functions which need data from bags.

In [ ]:


def define_2d_rotation_matrix(radians):
    rotation_matrix = np.array([[np.cos(radians),-np.sin(radians)],
                            [np.sin(radians),np.cos(radians)]])
    return rotation_matrix


def rotate_and_translate_points_based_on_fixed_beacons(fixed_beacons_pose,X,Y):
    ## Marvelmind results were I passed the distance between the beacons

    

    radians_between_1_and_2=math.atan2(fixed_beacons_pose[1,1] - fixed_beacons_pose[0,1],fixed_beacons_pose[1,0] - fixed_beacons_pose[0,0])
    radians_correction = -radians_between_1_and_2 # you want to turn to the opposite side for the correction.
    rotation_matrix = define_2d_rotation_matrix(radians_correction)

    x_fixed_beacon = fixed_beacons_pose[:,0]
    y_fixed_beacon = fixed_beacons_pose[:,1]

  
    i=0
    x_corrected_fixed_beacon_pose = []
    y_corrected_fixed_beacon_pose = []
    for x,y,p in zip(x_fixed_beacon, y_fixed_beacon,point_array):
        current_point = np.array([x,y]) 
        [new_x,new_y] = rotation_matrix.dot(current_point)
        if i==0:
            translate_y = new_y
            translate_x = new_x
        new_y = new_y - translate_y
        new_x = new_x - translate_x
        x_corrected_fixed_beacon_pose.append(new_x)
        y_corrected_fixed_beacon_pose.append(new_y)
        i=1                                      

    corrected_mobile_beacon_pose_x = []
    corrected_mobile_beacon_pose_y = []
    for x,y,p in zip(X, Y,point_array):
        current_point = np.array([x,y]) 
        [new_x,new_y] = rotation_matrix.dot(current_point)
        new_y = new_y - translate_y
        new_x = new_x - translate_x                                    
        corrected_mobile_beacon_pose_x.append(new_x)
        corrected_mobile_beacon_pose_y.append(new_y)
    return [corrected_mobile_beacon_pose_x,corrected_mobile_beacon_pose_y]


def plot_difference_between_real_and_program_poses(correct_x,corrected_y):
    fig, ax = plt.subplots()
    ax.grid()
    for x,y,p in zip(X_real, Y_real,point_array):
        ax.annotate(f"real {p}:({x}, {y})", xy=(x,y), xytext=(x, y))
        ax.plot(x,y, 'bx') 

    for x,y,p in zip(correct_x, corrected_y,point_array):
        ax.annotate(f'{p}:({x:.2f}, {y:.2f})', xy=(x,y), xytext=(x, y))
        ax.plot(x,y, 'ro') 

    ax.set_title('Real versus marvelmind pose')
    ax.set_xlabel('x (meters)')
    ax.set_ylabel('y (meters)')
    plt.show()

def plot_error_between_real_and_program_pose(corrected_x, corrected_y,message):
    fig, ax = plt.subplots()
    point_error_array = []
    for i in range(len(point_array)):
        point_error_array.append(math.sqrt((X_real[i] - corrected_x[i])**2 + (Y_real[i] - corrected_y[i])**2) * 100) # calculate euclidian distance and convert to cm
        a = ax.bar(f"{point_array[i]}", point_error_array[-1])
        temp_point_error_array = [f"{point_error_array[-1]:.2f}"]
        ax.bar_label(a, labels=temp_point_error_array)

    ax.set_title('Error between real and marvelmind program'f"({message})")
    ax.set_xlabel('point')
    ax.set_ylabel('error (cm)')
    plt.show()
    return point_error_array


  

### Comparison between marvelmind program and real points.


##### Strategy
1. Create ortogonal axes from (0,0)
2. Compare real to ortogonal axes, see if there is rotation and displacement. Ideally points 1,2 and 4, should form a close orthogonal axes.
    1. Calculate rotation between fixed beacons positions 1 and 2.
    2. Use a rotation matrix to rotate the points -correction.
    3. Translate the points to make position 1 approximately the origin (0,0)
3. Compare marvelmind program (passing distance) to ortogonal axes.
    1. Apply the same principals used by real.





In [17]:
## Point 8.3 had results way outside of acceptable ranges, and points 7.1 and 7.5 just repeated the pose of the last point.
## Probably, the mobile beacon lost its signal in poses 7.1 and 7.5. 
ignore_points = ['7.1','7.5','8.3'] # Some points were ignored because they were outliersarray_of_lists_to_be_cut = [marvel_with_distance_positions,marvel_without_distance_positions,real_points]
array_of_lists_to_be_cut = [marvel_with_distance_positions,marvel_without_distance_positions,real_points]
for point_to_be_ignored in ignore_points:
    for list_to_be_cut in array_of_lists_to_be_cut:
      list_to_be_cut.pop(point_to_be_ignored,None)

In [36]:
plt.close('all')
fixed_beacons_pose = np.array([[0.004,0.005],[10.246,0.360],[10.097,4.107],[-0.140,3.740]])
[corrected_mobile_beacon_x_with_distance, corrected_mobile_beacon_y_with_distance] = rotate_and_translate_points_based_on_fixed_beacons(fixed_beacons_pose,X_with_distance,Y_with_distance)
point_error_array = plot_error_between_real_and_program_pose(corrected_mobile_beacon_x_with_distance,corrected_mobile_beacon_y_with_distance, "with distance")
point_error_mean = np.mean(np.array(point_error_array))

In [35]:
plt.close('all')
fixed_beacons_pose = np.array([[0.001,0.001],[10.112,0.270],[9.991,4.004],[-0.140,3.630]])
[corrected_mobile_beacon_x_without_distance,corrected_mobile_beacon_y_without_distance] = rotate_and_translate_points_based_on_fixed_beacons(fixed_beacons_pose,X_without_distance,Y_without_distance)
point_error_array2 = plot_error_between_real_and_program_pose(corrected_mobile_beacon_x_without_distance,corrected_mobile_beacon_y_without_distance, "without distance")
point_error_mean = np.mean(np.array(point_error_array2))

In [31]:

marvelmind_error_dictionary = {
    'without_distance': point_error_array2,
    'with_distance': point_error_array,
}
width = 0.6  # the width of the bars: can also be len(x) sequence


fig, ax = plt.subplots()
bottom = np.zeros(len(point_array))

for type_of_program, error_array in marvelmind_error_dictionary.items():
    p = ax.bar(point_array, error_array, width, label=type_of_program, bottom=bottom)
    bottom += error_array
    ax.bar_label(p, label_type='center')

ax.set_title('Error between marvelmind program passing or not distance between fixed beacons.')
ax.legend()

plt.show()

NameError: name 'point_error_array2' is not defined

In [38]:
diff_between_1_and_2 = np.subtract(np.array(point_error_array),np.array(point_error_array2))
fig, ax = plt.subplots()
for i in range(len(diff_between_1_and_2)):
    a = ax.bar(f"{point_array[i]}", diff_between_1_and_2[i])
    temp_point_error_array = [f"{diff_between_1_and_2[i]:.2f}"]
    ax.bar_label(a, labels=temp_point_error_array)

ax.set_title('Error between real and marvelmind program')
ax.set_xlabel('point')
ax.set_ylabel('error (cm)')
plt.show()



In [3]:
a = {}
a['1'] = {"x": 1, "y":2}
a['67'] = {"x": 4, "y":267}
a['a'] = {"x": 5, "y":111}
a['hj'] = {"x": 89, "y":8}

a.pop('hj', None)
print(a)

{'1': {'x': 1, 'y': 2}, '67': {'x': 4, 'y': 267}, 'a': {'x': 5, 'y': 111}}


transformar os dados em dicionários do tipo com cada posição sendo p
exemplo
real[8.1].x
with_distance[7.1].y
x,y,z